In [ ]:
!pip install kiwipiepy spacy
# !pip install langdetect
!python -m spacy download en_core_web_sm

import colorsys
import re
import pandas as pd
from collections import Counter
# from langdetect import detect
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords
import spacy
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 15.6 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.23.0-py3-none-any.whl size=88067826 sha256=a281517aab116fb6b0197beb776d8ac6984eb57b11d03ac97998f6752c3bc597
  Stored in directory: /root/.cache/pip/wheels/f2/94/da/ff88b4c2305cd1f3effc8001b5f42f16dc9931bcd84d1e77c3
Successfully built kiwipiepy_model
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Mounted at /content/drive


In [ ]:
# 0. Settings

# uploaded = files.upload()

Lyrics_CSV = "/content/drive/MyDrive/khu/capstone1/lyrics_for_test.csv"
title_column = "title"
lyrics_column = "lyrics"

# output_token_csv = "lyrics_tokens_long.csv"
# output_core_csv = "core_keywords_by_song.csv"

# Core_Keywords_CSV = "/content/drive/MyDrive/khu/capstone1/core_keywords_by_song.csv" # 핵심 단어 입력 파일 경로
Lexicon_CSV = "/content/drive/MyDrive/khu/capstone1/Korean-NRC-EmoLex.txt" # 처리된 kor nrc emolex

Output_Matched_CSV = "/content/drive/MyDrive/khu/capstone1/matched_keywords_emotions.csv" # 단어별 매칭 결과 출력 파일 이름
Output_Song_Scores_Color_CSV = "/content/drive/MyDrive/khu/capstone1/song_emotion_scores_color.csv" # 노래별 감정 점수 및 색 매칭 출력 파일 이름
Output_Song_Color_Summary_CSV = "/content/drive/MyDrive/khu/capstone1/song_color_summary.csv" # 노래별 색 매칭 요약 출력 파일 이름

# min_word_length = 2 # 지나치게 짧은 단어는 제외
# min_count_in_song = 2 # 너무 안 나오는 단어도 제외

allowed_pos = {"noun", "verb", "adjective"} # 명사 동사 형용사만 후보로 추출

lyrics_stopwords = {
    # Korean
    "나", "너", "우리", "그", "그녀", "저", "이", "것", "거", "진짜", "그냥", "너무", "정말", "약간", "음", "아", "그게", "와", "하", "사람", "남자", "여자", "아이"
    # English
    "yeah", "oh", "ooh", "uh", "ah", "la", "hey", "na", "da", "ra", "aha", "really", "um", "ha", "I", "me", "you", "us", "he", "she", "they", "her", "his", "them", "our", "this", "that", "such", "so", "girl", "boy", "man", "woman", "guy", "something", "Whoa"
} # 커스텀 stopwords 조정 필수

Negation_Words = {
    "안", "않", "못", "마", "아니", "말", "아니야", "아냐", "없"
    "not", "no", "never", "n't"
}

Negation_Target_Pos = {
    "noun", "verb", "adjective"
}

Negation_Scope= 2

Negation_Exception_Replacements = [
    (r"can\s*not\s+help\s+but", " "),
    (r"can't\s+help\s+but", " "),
    (r"cannot\s+help\s+but", " "),
    (r"can\s*not\s+help\s+", " "),
    (r"can't\s+help\s+", " "),
    (r"cannot\s+help\s+", " "),
]

Emotion_Cols = [
    "joy", "sadness", "anger", "fear", "trust", "disgust", "surprise", "anticipation"
] # NRC 8감정 컬럼 목록
Sentiment_Cols = [
    "positive", "negative"
] # NRC 긍정/부정 컬럼 목록
Score_Cols = Emotion_Cols + Sentiment_Cols # 전체 점수 컬럼 목록

In [ ]:
# 1. Load NLP Tools

kiwi = Kiwi()
ko_stopwords = Stopwords()
nlp_en = spacy.load("en_core_web_sm")

In [ ]:
# 2. Split Lyrics

def split_into_chunks(text, chunk_size=15):
  words = text.split()
  chunks = []

  for i in range(0, len(words), chunk_size):
    chunk = " ".join(words[i:i + chunk_size])
    if chunk.strip():
      chunks.append(chunk)

  return chunks

def split_lyrics(lyrics):
  text = str(lyrics)

  # 가능하면 줄 기준 분리 시도
  lines = [l.strip() for l in text.split("\n") if l.strip()]

  if len(lines) > 1:
    return lines

  # 문장부호 기준 분리 시도
  sentences = re.split(r"[.!?]+", text)
  sentences = [s.strip() for s in sentences if s.strip()]

  if len(sentences) > 1:
    return sentences

  # fallback: 적당한 사이즈의 chunk 기준 분할
  return split_into_chunks(text, chunk_size = 15)

In [ ]:
# 3. Language Detection

def guess_language(text):
  kor_chars = len(re.findall(r"[가-힣]", text)) # 한국어 파트 추출
  eng_chars = len(re.findall(r"[A-Za-z]", text)) # 영어 파트 추출

  if kor_chars > 0 and eng_chars > 0: # 한국어 영어 둘 다 있으면 믹스
    return "mixed"
  if kor_chars > 0: # 한국어만
    return "kor"
  if eng_chars > 0: # 영어만
    return "eng"

  return "other" # 그 외 언어만 있으면 other

In [ ]:
# pre-4 Korean Negation

def is_korean_negation(word, tag):
  if word in {"안", "못"} and tag == "MAG":
    return True

  if word in {"않", "말", "마"} and tag == "VX":
    return True

  if word in {"없"} and tag == "VA":
    return True

  if word in {"아니", "아니야", "아냐"}:
    return True

  return False

In [ ]:
# 4. Korean Processing

def process_korean(text):
  tokens = kiwi.tokenize( # 문장을 형태소 단위로 분리
      text,
      normalize_coda = True # 비정형 텍스트 정규화
  )

  results = []

  for token in tokens:
    word = token.form
    tag = token.tag

    # 품사, kiwi 기본 불용어 처리 전에 부정어 살리기
    if is_korean_negation(word, tag):
      results.append((word, "negation", "kor"))
      continue

    if tag in {"NNG", "NNP"}: # 일반명사, 고유명사
      pos = "noun"
    elif tag == "VV": # 동사
      pos = "verb"
    elif tag == "VA": # 형용사
      pos = "adjective"
    else:
      pos = "other"

    results.append((word, pos, "kor")) # (단어, 품사, 언어)

  return results

In [ ]:
# pre-5 English Negation

def is_english_negation(token):
  word = token.text.lower()

  if word in {"no", "not", "never", "n't"}:
    return True

  return False

In [ ]:
# 5. English Processing

def process_english(text):
  doc = nlp_en(text) # 문장 -> 토큰 + 품사 + 원형 + 기타 정보

  results = []

  for token in doc:
    raw_word = token.text.lower()

    if is_english_negation(token):
      results.append((raw_word, "negation", "eng"))
      continue

    if not token.is_alpha:
      results.append((raw_word, "other", "eng"))
      continue

    word = token.lemma_.lower() # 단어를 원형 & 소문자로 변환

    if token.pos_ in {"NOUN", "PROPN"}: # 일반명사, 고유명사
      pos = "noun"
    elif token.pos_ == "VERB": # 동사
      pos = "verb"
    elif token.pos_ == "ADJ": # 형용사
      pos = "adjective"
    else:
      pos = "other"

    results.append((word, pos, "eng")) # 단어, 품사, 언어

  return results

In [ ]:
# 6. Other Language Fallback

def process_other(text): # fallback 은 보존만
  words = text.split()

  return [
      (word, "unknown", "other")
      for word in words
  ]

In [ ]:
# 7. Mixed Language Processing

def process_mixed(text):
  results = []

  korean_part = " ".join(re.findall(r"[가-힣]+", text)) # 한국어 파트 추출
  english_part = " ".join(re.findall(r"[A-Za-z]+", text)) # 영어 파트 추출

  if korean_part: # 한국어 파트 처리
    results.extend(process_korean(korean_part))

  if english_part: # 영어 파트 처리
    results.extend(process_english(english_part))

  return results

In [ ]:
# 8. Analyze One Lyric

def analyze_lyrics(lyrics):
  segments = split_lyrics(lyrics) # 가사 적당히 분리
  all_tokens = []

  for segment in segments:
    lang = guess_language(segment) # 분리한 가사 조각마다 언어 판별

    if lang == "kor": # 한국어 처리
      all_tokens.extend(process_korean(segment))
    elif lang == "eng": # 영어 처리
      all_tokens.extend(process_english(segment))
    elif lang == "mixed": # 믹스 처리
      all_tokens.extend(process_mixed(segment))
    else: # 나머지 처리
      all_tokens.extend(process_other(segment))

  return all_tokens

In [ ]:
# Pre-9 Negation Exception Replacement

def remove_negation_exceptions(text):
  text = str(text)

  for pattern, replacement in Negation_Exception_Replacements:
    text = re.sub(
        pattern,
        replacement,
        text,
        flags=re.IGNORECASE
    )

  return text

In [ ]:
# 9. Build Token Table

def build_token_table(df):
  rows = []

  for _, row in df.iterrows(): # DataFrame 돌면서 title, lyrics 꺼내기
    title = row[title_column]
    lyrics = row[lyrics_column]
    lyrics = remove_negation_exceptions(lyrics)

    analyzed_tokens = analyze_lyrics(lyrics) # 가사 적당히 분리 후 처리

    for token_index, (word, pos, lang) in enumerate(analyzed_tokens):
      word = word.strip().lower() # 공백 처리, 대문자 -> 소문자

      rows.append({
          "title": title,
          "token_index": token_index,
          "word": word,
          "pos": pos,
          "lang": lang,
          "is_negated": False
      })

  return pd.DataFrame(rows) # DataFrame 반환

In [ ]:
# 10. Negation Marking

def mark_negated_tokens(token_df):
  token = token_df.copy()

  token["negation_hits"] = 0
  token["is_negated"] = False

  for title, group in token.groupby("title"):
    group = group.sort_values("token_index")

    neg_positions = group[
        group["word"].isin(Negation_Words)
    ]["token_index"].tolist()

    target_indices = group[
        group["pos"].isin(Negation_Target_Pos)
    ]["token_index"].tolist()

    for target_pos in target_indices:
      start = target_pos - Negation_Scope
      end = target_pos + Negation_Scope

      hits = sum(
          start <= neg_pos <= end
          for neg_pos in neg_positions
      )

      mask = (
          (token["title"] == title)
          & (token["token_index"] == target_pos)
      )

      token.loc[mask, "negation_hits"] = hits
      token.loc[mask, "is_negated"] = (hits % 2 == 1)

  return token

In [ ]:
# 11. Core Keyword Filtering

def filter_core_keywords(token_df):
  if token_df.empty:
    return token_df

  filtered = token_df.copy() # 원본 보존

  # other / negation 버리고 불용어도 drop
  filtered = filtered[filtered["pos"].isin(allowed_pos)]
  filtered = filtered[~filtered["word"].isin(lyrics_stopwords)]

  core_df = (
      filtered
      .groupby(["title", "word", "pos", "lang"], as_index=False)
      .agg(
          count=("word", "size"),
          negated_count=("is_negated", "sum")
      )
  )

  # core_df = core_df[core_df["count"] >= min_count_in_song]

  core_df = core_df.sort_values(
      ["title", "count"],
      ascending=[True, False]
  )

  return core_df

In [ ]:
# 핵심 단어 -> 감정 -> 색
'''
더미 사전 csv 형식은 NRC 기반 csv 와 동일
word, lang, pos, joy, sadness, anger, fear, trust, disgust, surprise, anticipation, positive, negative
매치 컬러는 yellow, blue, red, purple, green, brown, orange, pink, saturation & value
즉, HSV : Hue, Saturation, Value 사용
'''

In [ ]:
# NRC EmoLex Processing

def build_nrc_ko_en_lexicon(path):
  raw = pd.read_csv(path, sep="\t")

  en_df = raw.rename(columns={"English Word": "word"}).copy()
  en_df["lang"] = "eng"
  en_df["pos"] = "any"

  ko_df = raw.rename(columns={"Korean Word": "word"}).copy()
  ko_df["lang"] = "kor"
  ko_df["pos"] = "any"

  en_df = en_df[["word", "lang", "pos"] + Score_Cols]
  ko_df = ko_df[["word", "lang", "pos"] + Score_Cols]

  lexicon_df = pd.concat([en_df, ko_df], ignore_index=True)

  lexicon_df["word"] = lexicon_df["word"].astype(str).str.strip().str.lower()

  for col in Score_Cols:
    lexicon_df[col] = pd.to_numeric(
        lexicon_df[col],
        errors = "coerce"
    ).fillna(0).astype(float)

  lexicon_df = lexicon_df[
      (lexicon_df["word"] != "") &
      (lexicon_df["word"] != "nan")
  ]

  lexicon_df = lexicon_df[
      ~lexicon_df["word"].str.contains(" ", regex=False)
  ]

  return lexicon_df

In [ ]:
def expand_korean_lexicon_with_kiwi(lexicon_df):
  rows = []

  for _, row in lexicon_df.iterrows():
    rows.append(row.to_dict())

    if row["lang"] != "kor":
      continue

    tokens = process_korean(row["word"])

    for token_word, token_pos, token_lang in tokens:
      new_row = row.to_dict()
      new_row["word"] = token_word
      new_row["pos"] = "any"
      rows.append(new_row)

  expanded_df = pd.DataFrame(rows)

  expanded_df = expanded_df.drop_duplicates(
      subset=["word", "lang", "pos"],
      keep="first"
  )

  return expanded_df

In [ ]:
def remove_confliction_duplicates_flexible(lexicon_df, threshold=2):
  df = lexicon_df.copy()

  key_cols = ["word", "lang", "pos"]

  df["score_cum"] = df[Score_Cols].sum(axis=1)

  rows_to_keep = []

  for _, group in df.groupby(key_cols):
    # 완전히 같은 중복 포함해서 후보들
    unique_scores = group[Score_Cols].drop_duplicates()

    # score 조합이 하나뿐이면 하나만 유지
    if len(unique_scores) == 1:
      rows_to_keep.append(group.iloc[0])
      continue

    min_score = group["score_sum"].min()
    max_score = group["score_sum"].max()

    # 점수 차이가 threshold 이하이면 가장 보수적인 행 유지
    if max_score - min_score <= threshold:
      keep_row = group.sort_values("score_sum").iloc[0]
      rows_to_keep.append(keep_row)

    # 차이가 너무 크면 다 날리기
    else:
      continue

  clean_df = pd.DataFrame(rows_to_keep)
  clean_df = clean_df.drop(columns=["score_sum"], errors="ignore")

  return clean_df

In [ ]:
# core keywords CSV 처리

def load_core_keywords(core_1_df):
  df = core_1_df
  # df = pd.read_csv(path) # path의 CSV를 읽어서 DataFrame으로 변환

  required = {"title", "word", "pos", "lang", "count"} # 이 CSV에 반드시 있어야 하는 컬럼 목록
  missing = required - set(df.columns) # 필수이나 실제 CSV에는 없는 컬럼 찾기
  if missing: # 빠진 컬럼이 있다면 코드 멈추고 에러 메시지 출력
    raise ValueError(f"core keywords csv missing columns: {missing}")

  # str으로 바꾸고 공백 제거, 소문자 통일
  df["word"] = df["word"].astype(str).str.strip().str.lower()
  df["lang"] = df["lang"].astype(str).str.strip().str.lower()
  df["pos"] = df["pos"].astype(str).str.strip().str.lower()
  df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0)
  # 숫자로 바꾸고 이상값은 NaN으로 바꿔서 0으로 채움

  return df # 처리한 core keywords DataFrame 반환

In [ ]:
# emotion lexicon CSV 처리

def load_emotion_lexicon(lexicon_df):
  df = lexicon_df
  # df = pd.read_csv(path) # path의 CSV를 읽어서 DataFrame으로 변환

  required = {"word", "lang", "pos", *Score_Cols} # 이 CSV에 반드시 있어야 하는 컬럼 목록, Score_Cols 내 원소들 포함
  missing = required - set(df.columns) # 필수이나 실제 CSV에는 없는 컬럼 찾기
  if missing: # 빠진 컬럼이 있다면 코드 멈추고 에러 메시지 출력
    raise ValueError(f"emotion lexicon csv missing columns: {missing}")
  # str으로 바꾸고 공백 제거, 소문자로 통일
  df["word"] = df["word"].astype(str).str.strip().str.lower()
  df["lang"] = df["lang"].astype(str).str.strip().str.lower()
  df["pos"] = df["pos"].astype(str).str.strip().str.lower()

  for col in Score_Cols: # Score_Cols 내 컬럼들을 순회하며 처리
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)
    # 숫자로 바꾸고 이상값은 NaN으로 바꿔서 0으로 채운 뒤 float으로 맞춤

  return df # 처리한 emotion lexicon DataFrame 반환

In [ ]:
# 핵심 단어 - 감정 매칭

def match_keywords(core_df, lexicon_df):
  core_df = core_df.copy() # 원본 보존 위해 copy
  lexicon_df = lexicon_df.copy()

  any_lexicon_df = lexicon_df.drop(columns=["pos"]) # pos 제외

  matched_df = core_df.merge( # DataFrame의 merge 기능
      any_lexicon_df,
      on=["word", "lang"], # word, lang이 같으면 같은 단어, pos는 제외
      how="left" # core_df 기준으로 lexicon_df를 merge : 사전에 없는 단어도 유지
  )

  matched_df["is_matched"] = ~matched_df[Score_Cols].isna().all(axis=1)
  # Score_Cols 확인해서 NaN이면, 즉, 사전과 매칭이 안 되면 False 되면 True
  matched_df[Score_Cols] = matched_df[Score_Cols].fillna(0) # NaN을 0으로 바꿈

  return matched_df # word - emotion 매칭한 DataFrame 반환

In [ ]:
# 부정어 근처 단어 처리

Positive_Cols = ["joy", "trust", "positive"]
Negative_Cols = ["sadness", "anger", "negative"]
Ambiguous_Cols = ["anticipation", "fear", "disgust"]

def flip_emotion_row(row):
  flipped = row.copy()

  has_positive = row[Positive_Cols].sum() > 0
  has_negative = row[Negative_Cols].sum() > 0

  if has_positive and not has_negative:
    flipped[Positive_Cols] = 0
    flipped[Negative_Cols] = 1

  elif has_negative and not has_positive:
    flipped[Negative_Cols] = 0
    flipped[Positive_Cols] = 1

  elif has_positive and has_negative:
    flipped[Positive_Cols + Negative_Cols] = 0

  for col in Ambiguous_Cols:
    if row[col] > 0:
      flipped[col] = 0

  return flipped

def apply_negation_adjustment(matched_df):
  df = matched_df.copy()

  if "negated_count" not in df.columns:
    df["negated_count"] = 0

  df["negated_count"] = pd.to_numeric(
      df["negated_count"],
      errors="coerce"
  ).fillna(0)

  df["count"] = pd.to_numeric(
      df["count"],
      errors="coerce"
  ).fillna(0)

  df["normal_count"] = df["count"] - df["negated_count"]
  df["normal_count"] = df["normal_count"].clip(lower=0)

  original_scores = df[Score_Cols].copy()
  flipped_scores = original_scores.apply(flip_emotion_row, axis=1)

  for col in Score_Cols:
    df[col] = (
        original_scores[col] * df["normal_count"]
        + flipped_scores[col] * df["negated_count"]
    )

  return df

In [ ]:
# 노래별 감정 점수 계산

def score_songs(matched_df):
  scored_df = matched_df.copy() # 원본 보존 위해 copy

  song_scores_df = (
      scored_df
      .groupby("title")[Score_Cols]
      .sum()
      .reset_index()
  ) # 곡 제목별로 묶어서 감정 및 긍정, 부정 점수 합산 테이블 생성

  # total 감정 점수
  song_scores_df["emotion_total"] = song_scores_df[Emotion_Cols].sum(axis=1)

  # 곡에서 positive / negative가 얼마나 잡혔는지
  song_scores_df["sentiment_total"] = (
      song_scores_df["positive"] + song_scores_df["negative"]
  )

  # 곡의 positive / negative 정도는 별개로
  song_scores_df["sentiment_balance"] = (
      song_scores_df["positive"] - song_scores_df["negative"]
  )

  # 곡의 positive / negative 비율
  song_scores_df["sentiment_ratio"] = (
      song_scores_df["sentiment_balance"] / song_scores_df["sentiment_total"]
  )

  song_scores_df["sentiment_ratio"] = (
      song_scores_df["sentiment_ratio"]
      .replace([float("inf"), float("-inf")], 0)
      .fillna(0)
  ) # +1 ~ -1 범위, +1이 완전 긍정, -1이 완전 부정 그리고 0이 중립

  # 주감정
  song_scores_df["primary_emotion"] = song_scores_df[Emotion_Cols].idxmax(axis=1)
  # 각 노래에서 8감정 중 가장 점수가 높은 감정을 대표 감정으로 설정하여 반환

  # 부감정
  def get_secondary_emotion(row):
    sorted_emotions = row[Emotion_Cols].sort_values(ascending=False)
    return sorted_emotions.index[1]

  song_scores_df["secondary_emotion"] = song_scores_df.apply(
      get_secondary_emotion,
      axis=1
  )

  # 8개 감정의 점수 합이 0인 곡을 찾아 "unmatched"를 대표 감정으로 설정
  zero_mask = song_scores_df["emotion_total"] == 0
  song_scores_df.loc[zero_mask, "primary_emotion"] = "unmatched"
  song_scores_df.loc[zero_mask, "secondary_emotion"] = "unmatched"

  # 감정 비율 컬럼
  for emotion in Emotion_Cols:
    song_scores_df[f"{emotion}_ratio"] = (
        song_scores_df[emotion] / song_scores_df["emotion_total"]
    )

  song_scores_df = song_scores_df.fillna(0)

  # 상위 감정 조합
  def get_emotion_combo(row, top_n=3):
    if row["emotion_total"] == 0:
      return "unmatched"
    sorted_emotions = row[Emotion_Cols].sort_values(ascending=False)
    top_emotions = [
        emotion for emotion in sorted_emotions.index[:top_n]
        if row[emotion] > 0
    ]

    return "+".join(top_emotions)

  song_scores_df["emotion_combo"] = song_scores_df.apply(
      get_emotion_combo,
      axis=1
  )

  return song_scores_df # 최종 노래별 감정 점수 테이블 반환

In [ ]:
# HSV 내부 계산
'''
NRC 기반 사전이라 형식은 동일
word, lang, pos, joy, sadness, anger, fear, trust, disgust, surprise, anticipation, positive, negative
매치 컬러는 yellow, blue, red, purple, green, brown, orange, pink, value
HSV : Hue, Saturation, Value 사용
'''

Emotion_HSV_Map = {
    "joy": (49, 0.90, 0.90),
    "sadness": (230, 0.94, 0.85),
    "anger": (0, 0.88, 0.85),
    "fear": (279, 0.90, 0.80),
    "trust": (131, 1.00, 0.75),
    "disgust": (43, 0.63, 0.65),
    "surprise": (27, 1.00, 0.90),
    "anticipation": (334, 0.65, 0.90),
    "unmatched": (0, 0.00, 0.60)
} # Hue, Saturation 고정하고 Value 만 조절

Emotion_Color_Name_Map = {
    "joy": "노란색",
    "sadness": "파란색",
    "anger": "빨간색",
    "fear": "보라색",
    "trust": "초록색",
    "disgust": "갈색",
    "surprise": "주황색",
    "anticipation": "분홍색",
    "unmatched": "회색"
}

def clamp(value, min_value=0.0, max_value=1.0):
  return max(min_value,min(value, max_value))

def hsv_to_hex(h, s, v): # h: 0~360, s/v: 0~1
  r, g, b = colorsys.hsv_to_rgb(h / 360, s, v)

  return "#{:02X}{:02X}{:02X}".format(
      int(r * 255),
      int(g * 255),
      int(b * 255)
  )

def adjust_hsv_by_sentiment(base_hsv, sentiment_ratio):
  h, s, v = base_hsv

  ratio = clamp(sentiment_ratio, -1, 1)

  v = v + (0.10 * ratio)
  v = clamp(sentiment_ratio, -1, 1)

  return h, s, v

def emotion_to_adjusted_hex(emotion, sentiment_ratio):
  base_hsv = Emotion_HSV_Map.get(
      emotion,
      Emotion_HSV_Map["unmatched"]
  )

  h, s, v = adjust_hsv_by_sentiment(
      base_hsv,
      sentiment_ratio
  )

  return hsv_to_hex(h, s, v)

def get_value_label(sentiment_ratio):
  if sentiment_ratio >= 0.35:
    return "밝은"
  elif sentiment_ratio <= -0.35:
    return "어두운"
  else:
    return ""

def emotion_to_color_text(emotion, sentiment_ratio):
  color_name = Emotion_Color_Name_Map.get(
      emotion,
      Emotion_Color_Name_Map["unmatched"]
  )
  value = get_value_label(sentiment_ratio)

  return color_name

In [ ]:
# 노래별 감정 - 색 매칭

def attach_color_columns(song_scores_df):
  song_scores_df["main_color_hex"] = song_scores_df.apply(
      lambda row: emotion_to_adjusted_hex(
          row["primary_emotion"],
          row["sentiment_ratio"]
      ),
      axis = 1
  )

  song_scores_df["main_color_text"] = song_scores_df.apply(
      lambda row: emotion_to_color_text(
          row["primary_emotion"],
          row["sentiment_ratio"]
      ),
      axis = 1
  )

  return song_scores_df # 최종 노래별 감정 - 색 매칭 테이블 반환

def build_palette_df(song_scores_df):
  palette_df = song_scores_df[
      [
          "title",
          "primary_emotion",
          "emotion_combo",
          "main_color_text",
          "main_color_hex",
          "sentiment_ratio"
      ]
  ].copy()

  return palette_df

In [ ]:
# CSV 처리 제외 DataFrame으로 바로 다루는 함수 통합

def make_core_df(lyrics):
  required_columns = {title_column, lyrics_column}
  missing = required_columns - set(lyrics.columns)

  if missing:
    raise ValueError(f"Missing columns: {missing}")

  token_df = build_token_table(lyrics)
  toekn_df = mark_negated_tokens(token_df)
  core_df = filter_core_keywords(token_df)

  return core_df

def processing_NRC_EmoLex(row_NRC):
  lexicon_df = build_nrc_ko_en_lexicon(row_NRC)
  lexicon_df = expand_korean_lexicon_with_kiwi(lexicon_df)
  lexicon_df = remove_confliction_duplicates_flexible(lexicon_df, threshold=2)

  return lexicon_df


def run_emotion_matching(core_df, lexicon_df):
  matched_df = match_keywords(core_df, lexicon_df) # 단어별 매칭 결과 확인용
  mathced_df = apply_negation_adjustment(matched_df) # 부정어 근처 단어 처리용
  song_scores_df = score_songs(matched_df) # 곡 - 감정 점수 매칭용
  song_scores_df = attach_color_columns(song_scores_df) # 색 컬럼 추가
  palette_df = build_palette_df(song_scores_df) # 요약 테이블 생성

  return matched_df, song_scores_df, palette_df

In [ ]:
# Main Pipeline

def main():
  df = pd.read_csv(Lyrics_CSV)
  core_df = make_core_df(df)
  lexicon_df = processing_NRC_EmoLex(Lexicon_CSV)
  matched_df, song_scores_df , palette_df = run_emotion_matching(
      core_df, lexicon_df)

  matched_df.to_csv(Output_Matched_CSV, index=False, encoding="utf-8-sig")
  song_scores_df.to_csv(Output_Song_Scores_Color_CSV, index=False, encoding="utf-8-sig")
  palette_df.to_csv(Output_Song_Color_Summary_CSV, index=False, encoding="utf-8-sig")

  print("Done!")

In [ ]:
if __name__ == "__main__":
  main()

Done!
